# 0. NumPy → PyTorch



## 1. Окружение и воспроизводимость

В Google Colab NumPy и PyTorch уже установлены. Фиксируем seed, чтобы примеры повторялись.


In [ ]:
import random
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("Устройство:", device)


## 2. NumPy: форма, тип и данные


In [ ]:
vector = np.array([1, 2, 3], dtype=np.float32)
matrix = np.arange(12).reshape(3, 4)
grid = np.linspace(0.0, 1.0, 5)
zeros = np.zeros((2, 3), dtype=np.float32)

for name, x in {"vector": vector, "matrix": matrix, "grid": grid, "zeros": zeros}.items():
    print(f"{name:>6}: shape={x.shape}, ndim={x.ndim}, dtype={x.dtype}")
    print(x, "\n")


Изображение обычно хранится в NumPy как **HWC**: высота, ширина, каналы. Значение пикселя может быть
`uint8` в диапазоне 0…255 или `float32` после подготовки данных.


In [ ]:
rng = np.random.default_rng(SEED)
image_hwc = rng.integers(0, 256, size=(4, 5, 3), dtype=np.uint8)
print("Изображение:", image_hwc.shape, image_hwc.dtype)
print("RGB первого пикселя:", image_hwc[0, 0])
print("Красный канал:\n", image_hwc[:, :, 0])


## Упражнение 1 — создание и dtype

<details>
<summary>Показать решение</summary>
np.linspace(0, 1, 12, dtype=np.float32).reshape(3, 4)
<details>

In [ ]:
# Создайте 3×4 float32-матрицу со значениями от 0 до 1 включительно.
ex1 =
assert ex1.shape == (3, 4) and ex1.dtype == np.float32
print(ex1)


## 3. Индексация, маски и view


In [ ]:
a = np.arange(12).reshape(3, 4)
print("a:\n", a)
print("Вторая строка:", a[1])
print("Последний столбец:", a[:, -1])
print("Центральный блок:\n", a[1:, 1:3])
print("Только чётные:", a[a % 2 == 0])


In [ ]:
original = np.arange(6)
view = original[1:4]       # срез обычно разделяет память с original
copy = original[1:4].copy()

view[0] = 99
copy[1] = -1
print("original:", original)  # изменился из-за view
print("view:    ", view)
print("copy:    ", copy)      # независим
print("shares_memory(original, view):", np.shares_memory(original, view))


## Упражнение 2 — индексация, маска и copy

Дана матрица NumPy, содержащая числа от 0 до 19.

Выполните две операции.

Часть 1. Используя булеву индексацию, получите из исходной матрицы все чётные элементы. Не используйте для этого циклы.

Часть 2. Выберите левый верхний фрагмент исходной матрицы размером 2 × 2 и создайте его независимую копию. Замените в копии все значения на -1.

После выполнения убедитесь, что изменение копии не повлияло на исходную матрицу.

<details>
<summary>Показать решение</summary>
ex2_source = np.arange(20).reshape(4, 5)
ex2_even = ex2_source[ex2_source % 2 == 0]
ex2_safe = ex2_source[:2, :2].copy()
ex2_safe[:] = -1
<details>

In [ ]:
ex2_source =
ex2_even =
ex2_safe =
ex2_safe[:] =
assert ex2_even.tolist() == list(range(0, 20, 2))
assert (ex2_source >= 0).all()  # copy не изменил источник
print("even:", ex2_even, "\nsource intact:\n", ex2_source)


## 4. Оси и broadcasting

Для матрицы формы `(rows, columns)`:

- `axis=0` сворачивает строки → один результат на столбец;
- `axis=1` сворачивает столбцы → один результат на строку.

Broadcasting сопоставляет размеры **справа налево**. Размеры совместимы, если они равны или один из них равен 1.


In [ ]:
x = np.array([[1., 2., 3.],
              [4., 5., 6.]])
print("mean all:", x.mean())
print("mean axis=0 (по столбцам):", x.mean(axis=0))
print("mean axis=1 (по строкам):  ", x.mean(axis=1))

column_means = x.mean(axis=0, keepdims=True)  # shape (1, 3)
centered = x - column_means                   # (2, 3) - (1, 3)
print("\ncolumn_means shape:", column_means.shape)
print("centered:\n", centered)
print("Проверка:", centered.mean(axis=0))


In [ ]:
# нормализация каждого RGB-канала.
images = rng.random((8, 32, 32, 3), dtype=np.float32)  # NHWC
mean = images.mean(axis=(0, 1, 2), keepdims=True)       # (1, 1, 1, 3)
std = images.std(axis=(0, 1, 2), keepdims=True)
normalized = (images - mean) / (std + 1e-7)

print("images:", images.shape, "mean:", mean.shape)
print("Среднее после нормализации:", normalized.mean(axis=(0, 1, 2)).round(5))
print("Std после нормализации:    ", normalized.std(axis=(0, 1, 2)).round(5))


## Упражнение 3 — axes и broadcasting

Дан трёхмерный массив NumPy формы: (2, 3, 4)

Для каждой группы элементов вдоль последней оси вычислите её среднее значение.

Затем вычтите соответствующее среднее из каждого элемента массива так, чтобы после преобразования среднее значение вдоль последней оси было приблизительно равно нулю.

<details>
<summary>Показать решение</summary>
ex3 = np.arange(24, dtype=np.float32).reshape(2, 3, 4)
means = ex3.mean(axis=-1, keepdims=True)
centered = ex3 - means
<details>

In [ ]:
ex3 = np.arange(24, dtype=np.float32).reshape(2, 3, 4)
means =
centered =
assert means.shape == (2, 3, 1)
np.testing.assert_allclose(centered.mean(axis=-1), 0, atol=1e-6)
print("input:", ex3.shape, "means:", means.shape, "result:", centered.shape)


## 5. Векторизация вместо Python-циклов


In [ ]:
values = np.arange(200_000, dtype=np.float64)

start = time.perf_counter()
loop_result = np.array([v * v + 2 * v for v in values])
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = values**2 + 2 * values
vector_time = time.perf_counter() - start

np.testing.assert_allclose(loop_result, vector_result)
print(f"Цикл: {loop_time:.4f}s | NumPy: {vector_time:.4f}s | ускорение ≈ {loop_time/vector_time:.1f}×")


## Упражнение 4 — векторизация

Даны две матрицы X и Y. Каждая строка матрицы представляет точку в двумерном пространстве.

Необходимо вычислить евклидово расстояние между каждой точкой из X и каждой точкой из Y.

Результатом должна быть матрица, в которой элемент с индексами [i, j] содержит расстояние между X[i] и Y[j].

<details>
<summary>Показать решение</summary>
np.sqrt(((X[:, None, :] - Y[None, :, :]) ** 2).sum(axis=-1))
<details>

In [ ]:
# Попарные евклидовы расстояния между строками X и Y без Python-цикла.
X = np.array([[0., 0.], [1., 1.], [2., 0.]])
Y = np.array([[1., 0.], [2., 2.]])
distances =
assert distances.shape == (3, 2)
print(distances)


## 6. NumPy и PyTorch

CPU-тензор из `torch.from_numpy` обычно разделяет память с исходным массивом. Для независимой копии используйте
`torch.tensor(array)` или `.clone()`. На GPU сначала нужен `.cpu()`.


In [ ]:
np_array = np.array([1., 2., 3.], dtype=np.float32)
tensor_shared = torch.from_numpy(np_array)
tensor_copy = torch.tensor(np_array)

np_array[0] = 100
print("NumPy:       ", np_array)
print("Shared tensor:", tensor_shared)
print("Copied tensor:", tensor_copy)

back_to_numpy = tensor_shared.numpy()
print("Обратно в NumPy:", back_to_numpy, back_to_numpy.dtype)


In [ ]:
# PyTorch чаще использует NCHW, а NumPy/PIL — NHWC.
batch_nhwc = np.zeros((8, 32, 32, 3), dtype=np.float32)
batch_nchw = torch.from_numpy(batch_nhwc).permute(0, 3, 1, 2)
print("NHWC:", batch_nhwc.shape, "→ NCHW:", tuple(batch_nchw.shape))
print("dtype:", batch_nchw.dtype, "device:", batch_nchw.device)


## Упражнение 5 — NumPy ↔ Torch и layout

Дано RGB-изображение в виде NumPy-массива формы: (16, 20, 3)

Выполните следующие действия:

1. Преобразуйте NumPy-массив в PyTorch-тензор.
2. Измените порядок осей с HWC на CHW, чтобы каналы RGB находились на первой оси.
3. Сделайте полученный тензор contiguous.
4. Преобразуйте данные обратно из CHW в HWC.
5. Верните результат в NumPy.

<details>
<summary>Показать решение</summary>
ex5_hwc = np.zeros((16, 20, 3), dtype=np.float32)
ex5_chw = torch.from_numpy(ex5_hwc).permute(2, 0, 1).contiguous()
ex5_back = ex5_chw.permute(1, 2, 0).numpy()
<details>

In [ ]:
ex5_hwc = np.zeros((16, 20, 3), dtype=np.float32)
ex5_chw =
ex5_back =
assert ex5_chw.shape == (3, 16, 20)
np.testing.assert_array_equal(ex5_hwc, ex5_back)
print("HWC", ex5_hwc.shape, "→ CHW", tuple(ex5_chw.shape))


## 7. Autograd: производная как вычисление


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1
y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item(), "(аналитически: 2x + 3 = 7)")

# Градиенты накапливаются: повторный backward прибавит ещё 7.
x.grad.zero_()
(x**2).backward()
print("После zero_(), d(x²)/dx =", x.grad.item())


## Упражнение 6 — autograd и gradient check
Рассмотрите функцию:

$$ f(w)=(w-4)^2 $$

для значения:

$$ w=1.5 $$

Вычислите производную функции двумя независимыми способами.

Способ 1 — PyTorch Autograd. Создайте тензор w с включённым вычислением градиентов, вычислите значение функции и вызовите backward(). Получите производную из w.grad.

Способ 2 — численное дифференцирование. Приблизительно вычислите производную методом центральной разности:

$$ f'(w)\approx \frac{f(w+\varepsilon)-f(w-\varepsilon)} {2\varepsilon} $$

Используйте небольшое значение \(\varepsilon\), например 1e-3.

Сравните два полученных значения.

<details>
<summary>Показать решение</summary>
w = torch.tensor(1.5, requires_grad=True)
objective = (w - 4) ** 2
objective.backward()
analytical = w.grad.item()
eps = 1e-3
numerical = (((w.detach() + eps - 4) ** 2 - (w.detach() - eps - 4) ** 2) / (2 * eps)).item()
<details>

In [ ]:
w = torch.tensor(1.5, requires_grad=True)
objective = (w - 4) ** 2

analytical =
numerical =
assert abs(analytical - numerical) < 1e-3
print(f"autograd={analytical:.4f}, finite difference={numerical:.4f}")


## 8. Минимальная нейросеть и полный training loop — 7 минут

Обучим MLP различать точки внутри и вне круга. Здесь есть все элементы настоящего обучения:
`Dataset → DataLoader → model → loss → backward → optimizer.step`.


In [ ]:
generator = torch.Generator().manual_seed(SEED)
points = torch.rand((1_000, 2), generator=generator) * 2 - 1
labels = ((points[:, 0]**2 + points[:, 1]**2) < 0.5).long()

loader = DataLoader(TensorDataset(points, labels), batch_size=64, shuffle=True, generator=generator)
model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2)).to(device)
loss_fn = nn.CrossEntropyLoss()  # получает logits, НЕ softmax
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)

for epoch in range(20):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()       # 1. очистить старые градиенты
        logits = model(xb)          # 2. forward
        loss = loss_fn(logits, yb)  # 3. ошибка
        loss.backward()             # 4. градиенты
        optimizer.step()            # 5. обновить параметры

    if epoch in {0, 4, 9, 19}:
        model.eval()
        with torch.inference_mode():
            pred = model(points.to(device)).argmax(dim=1).cpu()
        accuracy = (pred == labels).float().mean().item()
        print(f"epoch={epoch+1:02d} loss={loss.item():.4f} accuracy={accuracy:.3f}")


## Практика

1. Для `a = np.arange(24).reshape(2, 3, 4)` получите среднее по последней оси. Какой будет shape?
2. Нормализуйте каждую строку матрицы `m`, чтобы сумма строки стала 1. Используйте broadcasting, без цикла.
3. Измените MLP: добавьте ещё один скрытый слой и сравните точность.



In [ ]:
a = np.arange(24).reshape(2, 3, 4)
answer_1 =

m = np.array([[1., 1., 2.], [2., 3., 5.]])
answer_2 =

assert answer_1.shape == (2, 3)
np.testing.assert_allclose(answer_2.sum(axis=1), np.ones(2))
print("Готово. answer_1 shape:", answer_1.shape)
print("Суммы строк:", answer_2.sum(axis=1))


## Мини-квиз

1. Какой атрибут показывает тип элементов массива?
2. Совместимы ли для broadcasting формы `(8, 32, 32, 3)` и `(3,)`?
3. Почему изменение NumPy-среза может изменить исходный массив?
4. Что делает `requires_grad=True`?
5. Почему перед `CrossEntropyLoss` не нужен `softmax`?
6. Зачем вызывают `optimizer.zero_grad()`?


## Куда дальше и источники

- [TensorTonic: NumPy Sheet](https://www.tensortonic.com/study-plans/numpy-basics)
- [TensorTonic: PyTorch Sheet](https://www.tensortonic.com/study-plans/pytorch-basics)
- [NumPy: absolute basics](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [PyTorch: Learn the Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)
- [PyTorch: tensors](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
